[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/02c_prepare_your_data.ipynb)

# Prepare your own dataset — load, clean, split, format

> **Fine-tuning series — 02c of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. This notebook is self-contained: run **Setup**, then the rest.


## Setup (run first)

Self-contained: imports, model id, device flags, and a tiny inline dataset so this notebook runs on its own.

In [ ]:
# Environment check — record these versions with any results you report.
import importlib
for lib in ["torch", "transformers", "peft", "trl", "datasets",
            "bitsandbytes", "accelerate", "unsloth", "adapters"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{lib:14s} not installed ({type(e).__name__})")

In [ ]:
# !pip install -U transformers peft trl bitsandbytes datasets accelerate
# !pip install unsloth   # only for the Unsloth section (CUDA only)
import gc, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # small + fast; swap for any causal LM
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_OK = torch.cuda.is_available() and not BF16_OK   # fp16 needs a GPU
print("device:", device, "| bf16:", BF16_OK)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# (a) Instruction data -> for SFT / LoRA / QLoRA / Unsloth / instruction / prompt tuning
instructions = [
    {"instruction": "Give the capital of France.", "output": "Paris."},
    {"instruction": "What is 7 times 8?", "output": "56."},
    {"instruction": "Translate 'good morning' to Spanish.", "output": "Buenos días."},
    {"instruction": "Name the largest planet.", "output": "Jupiter."},
    {"instruction": "Define photosynthesis in one line.",
     "output": "Plants converting sunlight, water, and CO2 into energy and oxygen."},
] * 8   # repeat just to have enough steps in the demo

# (b) Preference data -> for DPO (prompt + chosen/rejected, no reward model)
preferences = [
    {"prompt": "Explain gravity to a child.",
     "chosen": "Gravity is the pull that brings things down to the ground.",
     "rejected": "Gravity is a tensor field obeying the Einstein equations."},
    {"prompt": "Recommend a book.",
     "chosen": "Try 'The Hobbit' — a fun, easy adventure.",
     "rejected": "Read whatever, I don't care."},
] * 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

sft_ds = Dataset.from_list([to_chat_text(e) for e in instructions])
pref_ds = Dataset.from_list(preferences)
print(sft_ds[0]["text"][:160])

In [ ]:
# Shared trainer imports + a default LoRA config reused by several sections.
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

## Prepare your own dataset

Every method notebook used fake inline data. Here's how to turn **your own** data into a
training set: load → inspect → clean → split → format. This covers SFT (the common case);
preference data is noted at the end.

### Load it (CSV / JSON / JSONL / Hub)

Your data usually has an input and a target. Below: a CSV in memory (swap for your file path),
plus the one-liners for JSONL and the Hugging Face Hub.

In [ ]:
import io, pandas as pd

# --- from a CSV file: pd.read_csv("mydata.csv") ---
csv_text = """instruction,output
What is the capital of France?,Paris.
2 plus 2?,4
Name a primary color.,Red.
,(blank input row to clean)
What is the capital of France?,Paris."""          # a blank + a duplicate to clean below
df = pd.read_csv(io.StringIO(csv_text))
print(df)

# other sources (uncomment as needed):
# df = pd.read_json("mydata.jsonl", lines=True)                # JSONL
# from datasets import load_dataset
# ds = load_dataset("tatsu-lab/alpaca", split="train[:200]")   # straight from the Hub

### Inspect & clean

Drop empty rows, strip whitespace, remove duplicates, filter out too-short/too-long examples.
Garbage in, garbage out — this step matters more than the method.

In [ ]:
df = df.dropna()
df["instruction"] = df["instruction"].str.strip()
df["output"] = df["output"].str.strip()
df = df[(df["instruction"] != "") & (df["output"] != "")]
df = df.drop_duplicates()
df = df[df["instruction"].str.len().between(3, 4000)]
print(f"{len(df)} clean rows"); print(df)

### Split train / validation / test

Train fits the model, validation watches for overfitting, test gives one final unbiased number.
80 / 10 / 10 is a fine default; shuffle with a fixed seed for reproducibility.

In [ ]:
from datasets import Dataset

ds = Dataset.from_pandas(df, preserve_index=False)
tmp = ds.train_test_split(test_size=0.2, seed=42)               # 80% train / 20% holdout
holdout = tmp["test"].train_test_split(test_size=0.5, seed=42)  # holdout -> val + test
train_ds, val_ds, test_ds = tmp["train"], holdout["train"], holdout["test"]
print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")

### Format to the chat template

The final step: render each row into the model's chat format (a `text` column the trainer
reads). Loss is later masked to the assistant's answer (instruction tuning).

In [ ]:
def to_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

train_text = train_ds.map(to_text, remove_columns=train_ds.column_names)
print(train_text[0]["text"])
# train_text drops straight into SFTTrainer(train_dataset=train_text, ...) from notebook 08.

### Preference data (for DPO / ORPO / reward)

Same idea, three columns instead of two:

```python
from datasets import Dataset
pref = Dataset.from_list([
    {"prompt": "Explain gravity simply.",
     "chosen": "It pulls things toward the ground.",
     "rejected": "It is a rank-2 tensor field."},
])
```
KTO wants `{prompt, completion, label: bool}`; reward modeling wants `{chosen, rejected}` full
texts. (See the data-formats table in Foundations.)

**Quality checklist:** consistent formatting · no test data leaked into train · diverse examples
· correct chat template · enough rows for the *kind* of change you want.